In [ ]:

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np
from tqdm import tqdm
import time
import urllib.request
import pandas as pd

#Set random seed for reproduction
random.seed(2023)
np.random.seed(2023)
torch.manual_seed(2023)

!pip install -U torchtext==0.6.0

#torchtext package
from torchtext import data


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#Data loading

urllib.request.urlretrieve("https://raw.githubusercontent.com/LawrenceDuan/IMDb-Review-Analysis/master/IMDb_Reviews.csv", filename="IMDb_Reviews.csv")
df = pd.read_csv('IMDb_Reviews.csv', encoding='latin1')
df.head()

In [ ]:
#Check the dataset size

print('The total number of samples : {}'.format(len(df)))

In [ ]:
#Split the dataset into train and test dataset

train_df = df[:30000]
test_df = df[-20000:]

train_df.to_csv("train_data.csv", index=False)
test_df.to_csv("test_data.csv", index=False)

In [ ]:
# Text preprocessing using Field method

TEXT = data.Field(sequential=True, # Whether the datatype represents sequential data
                  use_vocab=True, # Whether to make a Vocabulary
                  tokenize=str.split, # The function used to tokenize strings using this field into sequential examples
                  lower=True, # Whether to lowercase the text
                  batch_first=True,
                  fix_length=200)   # Note that the length of the sentences are all different.
                                    # If the length of a sentence is longer than 200, the length is reduced to 200 (remove the back part). And if it is shorter than 200, the length is made 200 by padding with 0.

LABEL = data.Field(sequential=False,
                   batch_first=False,
                   is_target=True) #

In [ ]:
train_data, test_data = data.TabularDataset.splits(
        path='.', train='train_data.csv', test='test_data.csv', format='csv',
        fields=[('text', TEXT), ('label', LABEL)], skip_header=True)

In [ ]:
print(vars(train_data[1]))

In [ ]:
TEXT.build_vocab(train_data, min_freq=5) # Create vocabulary # Here, add only words that appear at least "min_freq (5)" times in the trainset to the vocabulary # Words that appear less than five times are replaced with the token <unk> in the sense of Unknown.
LABEL.build_vocab(train_data)

In [ ]:
vocab_size = len(TEXT.vocab)
num_classes = 2
print('The size of vocabulary : {}'.format(vocab_size))
print('The number of classes : {}'.format(num_classes))

In [ ]:
TEXT.vocab.stoi
# torchtest assigns 0 to <unk> token and 1 to <pad> token.
# <unk> is used to represent words that is not in the vocabulary.
# <pad> is used for padding to match the fixed length.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 64

train_iter, test_iter = data.BucketIterator.splits(
        (train_data, test_data), batch_size=batch_size,
        shuffle=True, repeat=False)

batch = next(iter(train_iter))

In [ ]:
print(batch.text)

In [ ]:
batch = next(iter(train_iter))
print('The shape of a batch is :', batch.text.shape)

In [ ]:
train_data, val_data = train_data.split(split_ratio=0.8)

train_iter, val_iter, test_iter = data.BucketIterator.splits(
        (train_data, val_data, test_data), batch_size=batch_size,
        shuffle=True, repeat=False, sort=False)

In [ ]:
import math

class MyLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(MyLSTMCell, self).__init__()
        # fill in the blank --------------------------------------
        self.hidden_size = hidden_size
        self.xh2gates = nn.Linear(input_size + hidden_size, 4 * hidden_size)
        # --------------------------------------------------------
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.hidden_size) if self.hidden_size > 0 else 0
        for weight in self.parameters():
            nn.init.uniform_(weight, -stdv, stdv)

    def forward(self, input, hidden_states):
        # fill in the blank --------------------------------------
        # [Citation of AI content] - This part is significantly generated by AI.
        h_prev, c_prev = hidden_states                  # (B, H)
        combined = torch.cat([input, h_prev], dim=1)    # (B, X + H)

        gates = self.xh2gates(combined)                 # (B, 4H)
        i, f, o, g = gates.chunk(4, dim=1)              # (B, H)

        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)

        c = f * c_prev + i * g
        h = o * torch.tanh(c)

        return h, c
        # --------------------------------------------------------

In [ ]:
# Make an lstm model that performs the above task (refer to the above figure).
# Note that batch-wise operation is performed. So check the shape of output at each stage.

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from tqdm import tqdm

import pdb


class MyLSTM(nn.Module):
    def __init__(self, num_layers, hidden_size, n_vocab, embed_dim, num_classes, device, dropout_p=0.5):
        '''
        num_layers: the number of recurrent layers (Here, we just set this to 1.)
        hidden_size: the number of features in the hidden state
        n_vocab: the size of vocabulary
        embed_dim: the dimension of embedding vectors
        num_classes: the number of classes

        '''
        super(MyLSTM, self).__init__()

        self.hidden_size = hidden_size
        self.device = device
        self.embed = nn.Embedding(n_vocab, embed_dim)   # Define an embedding layer. The shape of lookup table is (n_vocab, embed_dim).
        self.dropout = nn.Dropout(dropout_p)
        self.lstmcell =  MyLSTMCell(embed_dim, self.hidden_size)
        self.out = nn.Linear(self.hidden_size, num_classes)

    def forward(self, x):
        x = self.embed(x) # Word embedding is performed. Therefore the shape of input x becomes (batch_size, sequence_length, embed_dim)
        h, c = self._init_state(batch_size=x.size(0)) # Initialization of hidden and cell state.
        sequence_length = x.shape[1]

        hidden = []

        # fill in the blank --------------------------------------------------------------
        '''
        1) Embedding vectors corresponding to words are sequentially entered into the LSTMCell one by one, and 'h (hidden state)' and 'c (cell state)' are iteratively updated.
        Note that: Store the hidden state of all time steps in the 'hidden' defined as list above and make the 'hidden' (list) into a tensor type using `torch.stack`
        2) Then, you should take only the last hidden state and pass it through the dropout layer, and finally use the fc layer to perform classification.
        '''
        for t in range(sequence_length):
            x_t = x[:, t, :]
            h, c = self.lstmcell(x_t, (h, c))
            hidden.append(h)

        hidden = torch.stack(hidden, dim=1) # (B, T, H)
        h_last = hidden[:, -1, :]

        dropped = self.dropout(h_last)
        out = self.out(dropped)             # (B, num_classes)
        # --------------------------------------------------------------------------------

        return out # Shape: (batch size, number of classes)

    def _init_state(self, batch_size=1):
        h = torch.zeros((batch_size, self.hidden_size)).to(self.device)
        c = torch.zeros((batch_size, self.hidden_size)).to(self.device)
        return h, c


In [ ]:
# Hyper-paramter settings
lr = 0.001
epochs = 5                                                                       # If you increase the number of epochs, you can see the higher performance.
criterion = nn.CrossEntropyLoss(reduction='sum')


#Set random seed for reproduction
random.seed(2023)
np.random.seed(2023)
torch.manual_seed(2023)


# Define model and optimizer
model = MyLSTM(1, 256, vocab_size, 128, num_classes, device, 0.5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
def train(model, optimizer, train_iter, device, Criterion):
    """train model"""
    model.train()
    for b, batch in enumerate(train_iter):
        x, y = batch.text.to(device), batch.label.to(device)
        y.data.sub_(1)
        optimizer.zero_grad()
        output = model(x)
        loss = Criterion(output, y)
        loss.backward()
        optimizer.step()


def evaluate(model, val_iter, device, Criterion):
    """evaluate model"""
    model.eval()
    corrects, total_loss = 0, 0
    for batch in val_iter:
        x, y = batch.text.to(device), batch.label.to(device)
        y.data.sub_(1)
        output = model(x)
        loss = Criterion(output, y)
        total_loss += loss.item()
        corrects += (output.max(1)[1].view(y.size()).data == y.data).sum()


    size = len(val_iter.dataset)
    loss_avg = total_loss / size
    acc_avg = 100.0 * corrects / size
    return loss_avg, acc_avg

In [ ]:
#Model training
best_val_loss = None

for e in tqdm(range(1, epochs+1)):
    train(model, optimizer, train_iter, device, criterion )
    val_loss, val_accuracy = evaluate(model, val_iter, device, criterion)

    print("[Epoch: %d] val accuracy : %5.2f" % (e, val_accuracy))

    # Save best model
    if not best_val_loss or val_loss < best_val_loss:
        if not os.path.isdir("best_lstm_model"):
            os.makedirs("best_lstm_model")
        torch.save(model.state_dict(), './best_lstm_model/best_model.pt')
        best_val_loss = val_loss

In [ ]:
#Model evaluation

model.load_state_dict(torch.load('./best_lstm_model/best_model.pt'))
test_loss, test_acc = evaluate(model, test_iter, device, criterion)
print('test loss: %5.2f | text accuracy: %5.2f' % (test_loss, test_acc))

In [ ]:
# Make an lstm model with the attention mechanism that performs the above process.
# Note that batch-wise operation is performed. So check the shape of output at each stage.

# When implementing the attention mechanism, it is recommended to use broadcasting (the for loops may not be used). See https://numpy.org/doc/stable/user/basics.broadcasting.html for the details of broadcasting.
# If necessary, use unsqueeze() or squeeze() function to match the dimension for broadcasting.


import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from tqdm import tqdm

import pdb


class MyLSTM_att(nn.Module):
    def __init__(self, num_layers, hidden_size, n_vocab, embed_dim, num_classes, device, dropout_p=0.5):
        super(MyLSTM_att, self).__init__()


        self.hidden_size = hidden_size
        self.device = device
        self.embed = nn.Embedding(n_vocab, embed_dim)
        self.dropout = nn.Dropout(dropout_p)
        self.lstmcell =  MyLSTMCell(embed_dim, self.hidden_size)

        # fill in the blank --------------------------------------------------------------
        self.out = nn.Linear(2 * self.hidden_size, num_classes) # This is the FC layer for final prediction in the above figure.
        # --------------------------------------------------------------------------------


    def forward(self, x):
        x = self.embed(x) # shape: (batch_size, sequence_length, embed_dim)
        h, c = self._init_state(batch_size=x.size(0)) # initialization
        hidden = []
        sequence_length = x.shape[1]

        # fill in the blank --------------------------------------------------------------
        '''
        You should add the following attention mechnism to the model in task 1 (refer to the above figure):
        1) The last hidden state is given as a query, and based on this, the similarities with the hidden states in the previous step are measured using dot-product.
        2) Using a softmax function, the results are normalized and attention scores (α) for each hidden state are obtained.
        3) Based on the attention scores (α) of each hidden state, we take a weighted average of hidden states (except for the last) and then, obtain a context vector.
        4) The context vector is concatenated with the last hidden state and prediction is performed based on it.
        '''
        # [Citation of AI content] - This part is significantly generated by AI.
        for t in range(sequence_length):
            x_t = x[:, t, :]
            h, c = self.lstmcell(x_t, (h, c))
            hidden.append(h)

        hidden = torch.stack(hidden, dim=1) # (B, T, H)
        h_last = hidden[:, -1, :]           # (B, H)
        hidden = hidden[:, :-1, :]          # (B, T-1, H)

        query = h_last.unsqueeze(2)         # (B, H, 1)

        attention_scores = torch.bmm(hidden, query)             # (B, T-1, 1)
        attention_weights = F.softmax(attention_scores, dim=1)
        attention_weights = attention_weights.transpose(1, 2)   # (B, 1, T-1)

        h_context = torch.bmm(attention_weights, hidden).squeeze(1) # (B, H)

        combined = torch.cat([h_last, h_context], dim=1)    # (B, 2H)
        dropped = self.dropout(combined)
        out = self.out(dropped)             # (B, num_classes)
        # --------------------------------------------------------------------------------

        return out #shape: (batch size, number of classes)

    def _init_state(self, batch_size=1):
        h = torch.zeros((batch_size, self.hidden_size)).to(self.device)
        c = torch.zeros((batch_size, self.hidden_size)).to(self.device)
        return h, c

In [ ]:
lr = 0.001
epochs = 5  # If you increase the number of epochs, you can see the higher performance.
criterion = nn.CrossEntropyLoss()

#Set random seed for reproduction
random.seed(2023)
np.random.seed(2023)
torch.manual_seed(2023)

model = MyLSTM_att(1, 256, vocab_size, 128, num_classes, device, 0.5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
#Model training
best_val_loss = None



for e in tqdm(range(1, epochs+1)):
    train(model, optimizer, train_iter, device, criterion )
    val_loss, val_accuracy = evaluate(model, val_iter, device, criterion)

    print("[Epoch: %d] val accuracy : %5.2f" % (e, val_accuracy))

    if not best_val_loss or val_loss < best_val_loss:
        if not os.path.isdir("best_lstm_model_with_attention"):
            os.makedirs("best_lstm_model_with_attention")
        torch.save(model.state_dict(), './best_lstm_model_with_attention/best_model.pt')
        best_val_loss = val_loss

In [ ]:
#Model evaluation
model.load_state_dict(torch.load('./best_lstm_model_with_attention/best_model.pt'))
test_loss, test_acc = evaluate(model, test_iter, device, criterion)
print('test loss: %5.2f | text accuracy: %5.2f' % (test_loss, test_acc))

In [ ]:
import numpy as np

# Bring a sample from test dataset
test_example = vars(test_data[16])
print('Test example: ', test_example['text'])

In [ ]:
# Check the result with a test sample using our trained model

index_vector = []

for text in test_example['text']:
    index_vector.append(TEXT.vocab.stoi[text])

index_vector_numpy = np.array(index_vector)
index_vector_tensor = torch.from_numpy(index_vector_numpy)
index_vector_tensor = index_vector_tensor.view(1,index_vector_tensor.size(0))

prediction = model(index_vector_tensor.to(device)).max(1)[1].item() # make a prediction through our trained model
prediction = 'Positive review' if prediction==1 else 'Negative review'
Ground_truth = 'Positive review' if test_example['label']==1 else 'Negative review'

# Compare the result with groundtruth
print('The prediction of our trained model on the test example:', prediction)
print('The true label of the test example:', Ground_truth)